# 5 · How solid are these results?

Notebooks 02 and 03 report one confounded sample and one train/test split. This notebook repeats both, tests whether the uplift ranking is distinguishable from noise, and removes the selection bias from the budget optimum.

In [1]:
import warnings

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from uplift_causal.data import load_hillstrom
from uplift_causal.estimate import compare
from uplift_causal.evaluate_uplift import (
    blp,
    diff_means,
    gates,
    honest_optimum,
    qini_coefficient,
    qini_diff_ci,
    rate,
    segment_table,
    segments,
    segments_fdr,
    uplift_oob,
)
from uplift_causal.learners import baseline, cross_fit, fit_predict
from uplift_causal.montecarlo import RESULTS, WEIGHTS, summarize
from uplift_causal.plots import gates_plot, montecarlo_plot, save
from uplift_causal.sensitivity import e_value, robustness_value
from uplift_causal.simulate_bias import simulate_bias

warnings.filterwarnings("ignore")

/Users/p.schmidt-petter/Desktop/dev/dev/uplift-causal-marketing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Estimators over 200 confounded samples

`python -m uplift_causal.montecarlo` redraws the confounded sample 200 times per scenario and runs every estimator on each.
*linear*: the propensity is logistic in recency and purchase history, so a logistic propensity model is correctly specified.
*nonlinear*: the propensity depends on the squares of both, which a logistic model in the raw covariates cannot represent.
Coverage is the share of 95 % intervals that contain the RCT effect; with 200 repetitions its standard error is about 1.5 points at 95 %.

In [2]:
rct = load_hillstrom()
truth = diff_means(rct.y, rct.t)[0]
runs = pd.read_csv(RESULTS / "montecarlo_runs.csv")
summarize(runs, truth).round(4)

bias      sd    rmse  coverage  coverage_se  \
scenario  estimator                                                    
linear    Naive        0.0384  0.0024  0.0385     0.000       0.0000   
          PS Matching -0.0011  0.0086  0.0087     0.885       0.0226   
          IPW          0.0019  0.0068  0.0071     0.970       0.0121   
          DML         -0.0022  0.0051  0.0055     0.970       0.0121   
          AIPW         0.0000  0.0069  0.0069     0.960       0.0139   
nonlinear Naive        0.0067  0.0027  0.0072     0.840       0.0259   
          PS Matching  0.0125  0.0048  0.0134     0.365       0.0340   
          IPW          0.0129  0.0028  0.0132     0.105       0.0217   
          DML         -0.0004  0.0051  0.0051     0.980       0.0099   
          AIPW        -0.0000  0.0060  0.0060     0.955       0.0147   

                       ci_width  reps  
scenario  estimator                    
linear    Naive          0.0178   200  
          PS Matching    0.0332   200  
          IPW            0.0290   200  
          DML            0.0243   200  
          AIPW           0.0289   200  
nonlinear Naive          0.0181   200  
          PS Matching    0.0215   200  
          IPW            0.0184   200  
          DML            0.0227   200  
          AIPW           0.0251   200

In [3]:
save(montecarlo_plot(runs, truth), "montecarlo", height=420)

## What if a confounder is not observed?

The adjustment only removes bias from confounders that are in the data. Dropping purchase history from the covariates shows what an unobserved confounder of that strength does.
The E-value states how strongly a hidden confounder would have to be associated with both mailing and visiting (as a risk ratio) to explain the adjusted effect away; the robustness value states the same as a share of residual variance.

In [4]:
obs = simulate_bias(rct, WEIGHTS, strength=1.5)
full, short = compare(obs), compare(obs.drop(columns="log_history"))
pd.concat({"all confounders": full, "history unobserved": short}).assign(
    bias=lambda d: d.est - truth
).round(4)

est      lo      hi    bias
all confounders    Naive        0.0868  0.0779  0.0957  0.0415
                   PS Matching  0.0551  0.0386  0.0700  0.0099
                   IPW          0.0531  0.0395  0.0671  0.0079
                   DML          0.0477  0.0357  0.0597  0.0025
                   AIPW         0.0554  0.0415  0.0694  0.0102
history unobserved Naive        0.0868  0.0779  0.0957  0.0415
                   PS Matching  0.0540  0.0123  0.0739  0.0088
                   IPW          0.0605  0.0482  0.0709  0.0153
                   DML          0.0576  0.0469  0.0683  0.0124
                   AIPW         0.0612  0.0495  0.0728  0.0159

In [5]:
est, lo, hi = full.loc["DML"]
base = obs.y.mean() - est * obs.t.mean()
se = (hi - lo) / (2 * 1.96)
pd.Series({
    "risk ratio": (base + est) / base,
    "E-value": e_value((base + est) / base),
    "E-value, lower CI bound": e_value((base + lo) / base),
    "robustness value": robustness_value(est, se, len(obs) - obs.shape[1]),
}).round(3)

risk ratio                 1.461
E-value                    2.282
E-value, lower CI bound    2.027
robustness value           0.052
dtype: float64

## Uplift ranking over repeated cross-fitting

Five-fold cross-fitting, repeated five times: every customer gets an out-of-fold score in each repetition and the Qini coefficient uses all customers instead of one 40 % test set.

In [6]:
X = rct.drop(columns=["t", "y"])
names = ["S-Learner", "T-Learner", "X-Learner", "DR-Learner", "Causal Forest"]
oof = cross_fit(X, rct.t, rct.y, names, n_splits=5, repeats=5)
qini = pd.DataFrame({k: [qini_coefficient(rct.y, rct.t, s) for s in v] for k, v in oof.items()})
qini.agg(["median", "min", "max"]).T.round(4)

,median,min,max
S-Learner,0.0552,0.0536,0.0596
T-Learner,0.0414,0.0399,0.0449
X-Learner,0.0401,0.0393,0.0460
DR-Learner,0.0409,0.0386,0.0438
Causal Forest,0.0574,0.0512,0.0613


Paired comparison against the best learner: the bootstrap uses the same resampling weights for both rankings, so the interval is for the difference.

In [7]:
best = qini.median().idxmax()
pd.DataFrame(
    {k: qini_diff_ci(rct.y, rct.t, oof[best][0], oof[k][0]) for k in names if k != best},
    index=["qini_diff", "lo", "hi"],
).T.round(4)

,qini_diff,lo,hi
S-Learner,0.0044,-0.0042,0.0135
T-Learner,0.0167,0.0055,0.0284
X-Learner,0.0152,0.0048,0.0260
DR-Learner,0.0172,0.0055,0.0284


## Is there heterogeneity at all?

*BLP slope*: regress the outcome on treatment and treatment × centered τ̂. A slope of 1 means the predicted effects are calibrated, 0 means they carry no information.
*AUTOC*: area under the targeting curve, the average gain of treating the top-ranked fraction over treating at random. Both are tested against zero on out-of-fold scores.

In [8]:
rows = {}
for k in names:
    slope = blp(rct.y, rct.t, oof[k][0]).loc["heterogeneity"]
    rows[k] = (*slope, *rate(rct.y, rct.t, oof[k][0]))
pd.DataFrame.from_dict(
    rows, orient="index", columns=["blp_slope", "slope_se", "slope_p", "autoc", "autoc_se", "autoc_p"]
).round(4)

,blp_slope,slope_se,slope_p,autoc,autoc_se,autoc_p
S-Learner,0.4926,0.0772,0.0,0.0196,0.0038,0.0
T-Learner,0.2303,0.0459,0.0,0.0175,0.0036,0.0
X-Learner,0.2692,0.0521,0.0,0.0183,0.0037,0.0
DR-Learner,0.2161,0.0436,0.0,0.0163,0.0037,0.0
Causal Forest,0.7299,0.0925,0.0,0.0200,0.0036,0.0


In [9]:
groups = gates(rct.y, rct.t, oof[best][0])
save(gates_plot(groups, truth), "gates", height=420)

In [10]:
groups.round(4)

,predicted,uplift,lo,hi
1,-0.0005,0.0231,0.0098,0.0364
2,0.0174,0.0052,-0.0076,0.0181
3,0.0454,0.0367,0.0216,0.0517
4,0.0706,0.0940,0.0795,0.1086
5,0.0919,0.0669,0.0521,0.0817


## Segments that survive a multiple-testing correction

Notebook 03 assigns segments from point estimates. Here a customer is a *Persuadable* or *Sleeping Dog* only if the causal forest's effect is significantly different from zero at a false discovery rate of 10 % (Benjamini-Hochberg).

In [11]:
train, test = train_test_split(rct, test_size=0.4, stratify=rct.t * 2 + rct.y, random_state=0)
X_train, X_test = train.drop(columns=["t", "y"]), test.drop(columns=["t", "y"])
models, tau = fit_predict(X_train, train.t, train.y, X_test, names=["T-Learner", "Causal Forest"])
effect = np.ravel(tau["Causal Forest"])
p_values = np.ravel(models["Causal Forest"].effect_inference(X_test).pvalue())
mu0 = baseline(models["T-Learner"], X_test)
threshold, delta = train.y[train.t == 0].mean(), diff_means(train.y, train.t)[0] / 2
point = segments(mu0, effect, threshold, delta)
fdr = segments_fdr(mu0, effect, p_values, threshold, q=0.1)
pd.crosstab(pd.Series(point, name="point estimate"), pd.Series(fdr, name="FDR 10 %"), normalize="all").round(3)

FDR 10 %,Lost Causes,Persuadables,Sleeping Dogs,Sure Things
point estimate,,,,
Lost Causes,0.164,0.005,0.0,0.000
Persuadables,0.049,0.541,0.0,0.081
Sleeping Dogs,0.000,0.000,0.0,0.000
Sure Things,0.000,0.001,0.0,0.159


In [12]:
segment_table(test.y, test.t, fdr).round(4)

,share,uplift,lo,hi
Persuadables,0.5460,0.0687,0.0548,0.0826
Sure Things,0.2404,0.0155,-0.0061,0.0372
Lost Causes,0.2134,0.0175,-0.0005,0.0356
Sleeping Dogs,0.0001,0.0000,NaN,NaN


## Budget optimum without selection bias

The budget calculator picks the target share with the highest estimated profit. The maximum of a noisy curve overstates what that share will earn. For each bootstrap sample the share is chosen on the sample and scored on the customers left out of it. Values are profit per customer at €5 per incremental visit.

In [13]:
scores = pd.read_parquet("../app/scores_hillstrom.parquet")
bags = uplift_oob(scores.y, scores.t, scores["Causal Forest"])
rows = {}
for cost in (0.05, 0.10, 0.20, 0.25):
    out = honest_optimum(*bags, value=5.0, cost=cost)
    rows[cost] = (out["naive"], out["honest"], *out["share"])
pd.DataFrame.from_dict(
    rows, orient="index", columns=["naive", "corrected", "share_lo", "share_median", "share_hi"]
).rename_axis("cost per contact").round(4)

,naive,corrected,share_lo,share_median,share_hi
cost per contact,,,,,
0.05,0.1843,0.1647,0.5200,0.93,1.00
0.10,0.1480,0.1288,0.4995,0.59,0.99
0.20,0.0921,0.0792,0.4100,0.52,0.62
0.25,0.0672,0.0513,0.3200,0.51,0.59
